# preprocess.py
Builds the Boston ZIP-month housing panel from Zillow and FRED data.

In [1]:
"""Build a Boston ZIP-month housing panel from Zillow and FRED data and save it as boston_housing_preprocessed.csv."""
from pathlib import Path
import pandas as pd
BOSTON_ZIP_METRO = "Boston-Cambridge-Newton, MA-NH"
BOSTON_METRO_NAME = "Boston, MA"
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_FILE = PROJECT_ROOT / "outputs" / "boston_housing_preprocessed.csv"

In [2]:
def melt_zillow(df, value_name):
    id_cols = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName", "State", "City", "Metro", "CountyName"]
    value_cols = [c for c in df.columns if c not in id_cols]
    out = df.melt(id_vars=id_cols, value_vars=value_cols, var_name="date", value_name=value_name)
    out["date"] = pd.to_datetime(out["date"]).dt.to_period("M").dt.to_timestamp("M")
    out[value_name] = pd.to_numeric(out[value_name], errors="coerce")
    return out


def classify_location_tier(zip_code):
    inner_suburb_set = {"02138", "02139", "02140", "02141", "02142", "02143", "02144", "02145", "02445", "02446", "02148", "02149", "02472", "02474", "02155", "02150"}
    if zip_code in inner_suburb_set:
        return "inner_suburb"
    if zip_code.startswith("021"):
        return "boston_city"
    return "outer_suburb"


def load_fred_monthly(filename, value_col, resample=False):
    path = Path(filename)
    if not path.exists():
        print(f"Missing {filename}; skipping")
        return None
    df = pd.read_csv(path)
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df = df[["observation_date", value_col]].dropna().rename(columns={"observation_date": "date"})
    if resample:
        df = df.set_index("date").resample("ME").mean().reset_index()
    df["date"] = df["date"].dt.to_period("M").dt.to_timestamp("M")
    return df


def load_metro_feature(filename, value_name):
    path = Path(filename)
    if not path.exists():
        print(f"Missing {filename}; skipping")
        return None
    metro = pd.read_csv(path)
    metro = metro[metro["RegionName"] == BOSTON_METRO_NAME].copy()
    id_cols = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]
    value_cols = [c for c in metro.columns if c not in id_cols]
    out = metro.melt(id_vars=id_cols, value_vars=value_cols, var_name="date", value_name=value_name)
    out["date"] = pd.to_datetime(out["date"]).dt.to_period("M").dt.to_timestamp("M")
    out[value_name] = pd.to_numeric(out[value_name], errors="coerce")
    return out[["date", value_name]].drop_duplicates("date")

In [3]:
def main():
    zhvi_path = DATA_DIR / "Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
    zori_path = DATA_DIR / "Zip_zori_uc_sfrcondomfr_sm_month.csv"
    if not zhvi_path.exists() or not zori_path.exists():
        if not zhvi_path.exists():
            print("Missing Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv; cannot continue")
        if not zori_path.exists():
            print("Missing Zip_zori_uc_sfrcondomfr_sm_month.csv; cannot continue")
        return

    zhvi = pd.read_csv(zhvi_path, dtype={"RegionName": str})
    zori = pd.read_csv(zori_path, dtype={"RegionName": str})
    zhvi = melt_zillow(zhvi[zhvi["Metro"] == BOSTON_ZIP_METRO].copy(), "zhvi")
    zori = melt_zillow(zori[zori["Metro"] == BOSTON_ZIP_METRO].copy(), "zori")
    zhvi = zhvi.rename(columns={"RegionName": "zip_code"})
    zori = zori.rename(columns={"RegionName": "zip_code"})
    zhvi["zip_code"] = zhvi["zip_code"].str.zfill(5)
    zori["zip_code"] = zori["zip_code"].str.zfill(5)

    panel = zhvi[["zip_code", "date", "zhvi"]].merge(zori[["zip_code", "date", "zori"]], on=["zip_code", "date"], how="outer")
    panel["location_tier"] = panel["zip_code"].apply(classify_location_tier)

    for filename, col in [("Metro_invt_fs_uc_sfrcondo_sm_month.csv", "inventory"), ("Metro_mean_doz_pending_uc_sfrcondo_sm_month.csv", "days_to_pending"), ("Metro_new_homeowner_income_needed_downpayment_0.20_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv", "income_needed")]:
        feature = load_metro_feature(DATA_DIR / filename, col)
        if feature is not None:
            panel = panel.merge(feature, on="date", how="left")

    for filename, col, resample in [("MORTGAGE30US.csv", "MORTGAGE30US", True), ("UNRATE.csv", "UNRATE", False), ("BOST625URN.csv", "BOST625URN", False), ("CPIAUCSL.csv", "CPIAUCSL", False)]:
        fred = load_fred_monthly(DATA_DIR / filename, col, resample=resample)
        if fred is not None:
            panel = panel.merge(fred, on="date", how="left")

    panel = panel.sort_values(["zip_code", "date"]).drop_duplicates(["zip_code", "date"])
    panel["zhvi_yoy_pct"] = panel.groupby("zip_code")["zhvi"].pct_change(12) * 100
    panel["zori_yoy_pct"] = panel.groupby("zip_code")["zori"].pct_change(12) * 100
    OUTPUT_FILE.parent.mkdir(exist_ok=True)
    panel.to_csv(OUTPUT_FILE, index=False)
    print(f"Saved {OUTPUT_FILE} with {len(panel):,} rows")

In [4]:
if __name__ == "__main__":
    main()

Saved C:\Users\tanma\Desktop\DS4420\housingAffordability\outputs\boston_housing_preprocessed.csv with 85,136 rows
